In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
#pip install ipywidgets ## install the library
import ipywidgets as widgets
import os, glob


In [3]:
import os
import glob
import ipywidgets as widgets

import os
import glob
import math
import ipywidgets as widgets

def Export_folders_from_results(PATH):
    """Detect all subfolders inside PATH (simulation results)."""
    simulation_dirs = sorted([
        os.path.join(PATH, d)
        for d in os.listdir(PATH)
        if os.path.isdir(os.path.join(PATH, d))
    ])
    for s in simulation_dirs:
        print(s)
    return simulation_dirs  # opcional, útil si quieres usarlas después

def create_quad_viewer(png_directories, simulation_names=None):
    """
    Flexible viewer: shows any number of simulations (3, 4, 5, ...)
    in a grid layout with individual and synchronized controls.
    """

    all_png_files = []
    all_widgets = []
    all_sliders = []
    all_labels = []
    
    # Load images for each simulation
    for i, png_dir in enumerate(png_directories):
        png_files = sorted(glob.glob(os.path.join(png_dir, "*.png")))
        print(f"Found {len(png_files)} PNG files in {png_dir}")
        all_png_files.append(png_files)
        
        image_widget = widgets.Image(
            value=open(png_files[0], "rb").read(),
            format='png',
            width=400,
            height=300
        )
        all_widgets.append(image_widget)
        
        slider = widgets.IntSlider(
            value=0,
            min=0,
            max=len(png_files)-1,
            step=1,
            description=f'{simulation_names[i]}:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='400px')
        )
        all_sliders.append(slider)
        
        label = widgets.Label(
            value=f"Timestep 0/{len(png_files)-1}: {os.path.basename(png_files[0])}",
            layout=widgets.Layout(width='400px'),   
        )
        all_labels.append(label)
        
        # Update function (closure)
        def make_update_function(idx):
            def update_image(change):
                frame_idx = change['new']
                files = all_png_files[idx]
                with open(files[frame_idx], "rb") as f:
                    all_widgets[idx].value = f.read()
                all_labels[idx].value = f"Timestep {frame_idx}/{len(files)-1}: {os.path.basename(files[frame_idx])}"
            return update_image
        slider.observe(make_update_function(i), names='value')
    
    # Master control slider
    master_slider = widgets.IntSlider(
        value=0,
        min=0,
        max=min([len(files) for files in all_png_files])-1,
        step=1,
        description='Sync All:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='100%')
    )

    def sync_all(change):
        frame_idx = change['new']
        for slider in all_sliders:
            if frame_idx <= slider.max:
                slider.value = frame_idx
    master_slider.observe(sync_all, names='value')
    
    rows = []
    for i in range(0, len(all_widgets), 2):
        imgs = all_widgets[i:i+2]
        lbls = all_labels[i:i+2]
        slds = all_sliders[i:i+2]
        rows.append(widgets.HBox(lbls,layout=widgets.Layout(justify_content='center'),    width='100%',height='auto'))
        rows.append(widgets.HBox(imgs,layout=widgets.Layout(justify_content='center'),   width='100%', height='auto'))
        rows.append(widgets.HBox(slds,layout=widgets.Layout(justify_content='center'),   width='100%', height='auto'))

    viewer = widgets.VBox([
        widgets.HTML("<h3>Timesteps</h3>"),
        master_slider,
        widgets.HTML("<h3>Simulations</h3>"),
        *rows
    ])
    
    return viewer
 
 
 
from PIL import Image
import matplotlib.pyplot as plt
def plot_timesteps_images(folder, start, end, step=10, ncols=3, figsize=(12, 6), pad=3):
    """
    Displays a grid of PNG images for selected timesteps (with zero-padded filenames).

    Parameters
    ----------
    folder : str
        Directory where images are stored (named like 000.png, 001.png, ...).
    start : int
        First timestep to display.
    end : int
        Last timestep to display.
    step : int, optional
        Step interval between images (default = 10).
    ncols : int, optional
        Number of columns in the plot grid.
    figsize : tuple, optional
        Figure size for matplotlib.
    pad : int, optional
        Number of zero-padding digits (e.g., 3 → 000, 001, 020).
    """
    timesteps = list(range(start, end + 1, step))
    nrows = (len(timesteps) + ncols - 1) // ncols
    
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    axes = axes.flatten()
    
    for i, t in enumerate(timesteps):
        filename = os.path.join(folder, f"{t:0{pad}d}.png")
        if os.path.exists(filename):
            img = Image.open(filename)
            axes[i].imshow(img)
            axes[i].set_title(f"Timestep {t}")
            axes[i].axis("off")
        else:
            axes[i].text(0.5, 0.5, f"Missing\n{t:0{pad}d}.png", ha='center', va='center')
            axes[i].axis("off")
    
    # Hide extra axes
    for j in range(i + 1, len(axes)):
        axes[j].axis("off")
    
    plt.tight_layout()
    plt.show()

In [4]:
PATH = "/scratch/flore0a/Dataset_test2/Cases"  # Folder where all sims are located
Directorio_results = Export_folders_from_results(PATH)
## add "/Simulation2D" in the Directorio_results list for each simulation
Directorio_results = [d + "/Simulation2D/merged" for d in Directorio_results]
Directorio_results = Directorio_results[:-1]


/scratch/flore0a/Dataset_test2/Cases/D2.5_V2
/scratch/flore0a/Dataset_test2/Cases/D2.5_V2p5
/scratch/flore0a/Dataset_test2/Cases/D2.5_V3
/scratch/flore0a/Dataset_test2/Cases/D2.5_V3p5
/scratch/flore0a/Dataset_test2/Cases/D2.5_V4
/scratch/flore0a/Dataset_test2/Cases/D2.5_V4p5
/scratch/flore0a/Dataset_test2/Cases/D2.5_V5
/scratch/flore0a/Dataset_test2/Cases/D2.5_V5p5
/scratch/flore0a/Dataset_test2/Cases/D2.5_V6
/scratch/flore0a/Dataset_test2/Cases/D2.5_V6p5
/scratch/flore0a/Dataset_test2/Cases/D2_V2
/scratch/flore0a/Dataset_test2/Cases/D2_V2p5
/scratch/flore0a/Dataset_test2/Cases/D2_V3
/scratch/flore0a/Dataset_test2/Cases/D2_V3p5
/scratch/flore0a/Dataset_test2/Cases/D2_V4
/scratch/flore0a/Dataset_test2/Cases/D2_V4p5
/scratch/flore0a/Dataset_test2/Cases/D2_V5
/scratch/flore0a/Dataset_test2/Cases/D2_V5p5
/scratch/flore0a/Dataset_test2/Cases/D2_V6
/scratch/flore0a/Dataset_test2/Cases/D2_V6p5
/scratch/flore0a/Dataset_test2/Cases/D3.5_V2
/scratch/flore0a/Dataset_test2/Cases/D3.5_V2p5
/scratch

In [5]:
#how to remove the previos before /
name = [ os.path.basename(os.path.dirname(path)) for path in Directorio_results]
png_directory = [d + "/png/u_t" for d in Directorio_results] # add the /png folder


quad_viewer = create_quad_viewer(png_directory, name)
display(quad_viewer)

Found 32 PNG files in /scratch/flore0a/Dataset_test2/Cases/D2.5_V2/Simulation2D/merged/png/u_t
Found 24 PNG files in /scratch/flore0a/Dataset_test2/Cases/D2.5_V2p5/Simulation2D/merged/png/u_t
Found 32 PNG files in /scratch/flore0a/Dataset_test2/Cases/D2.5_V3/Simulation2D/merged/png/u_t
Found 12 PNG files in /scratch/flore0a/Dataset_test2/Cases/D2.5_V3p5/Simulation2D/merged/png/u_t
Found 10 PNG files in /scratch/flore0a/Dataset_test2/Cases/D2.5_V4/Simulation2D/merged/png/u_t
Found 12 PNG files in /scratch/flore0a/Dataset_test2/Cases/D2.5_V4p5/Simulation2D/merged/png/u_t
Found 10 PNG files in /scratch/flore0a/Dataset_test2/Cases/D2.5_V5/Simulation2D/merged/png/u_t
Found 14 PNG files in /scratch/flore0a/Dataset_test2/Cases/D2.5_V5p5/Simulation2D/merged/png/u_t
Found 12 PNG files in /scratch/flore0a/Dataset_test2/Cases/D2.5_V6/Simulation2D/merged/png/u_t
Found 8 PNG files in /scratch/flore0a/Dataset_test2/Cases/D2.5_V6p5/Simulation2D/merged/png/u_t
Found 12 PNG files in /scratch/flore0a/Da

In [6]:
import os
import glob
import ipywidgets as widgets

import os
import glob
import math
import ipywidgets as widgets

def create_grid_viewer(
    png_directories,
    simulation_names=None,
    ncols=10,
    image_width=110,
    image_height=85,
    show_individual_sliders=False
):
    """
    Compact grid viewer for many simulations.
    Better suited for 10x10-like overview layouts.
    """

    if simulation_names is None:
        simulation_names = [f"Sim {i+1}" for i in range(len(png_directories))]
    else:
        # make names shorter for display
        simulation_names = [
            name if len(name) <= 12 else name[:12] + "..."
            for name in simulation_names
        ]

    all_png_files = []
    all_widgets = []
    all_labels = []
    all_sliders = []
    simulation_boxes = []

    for i, png_dir in enumerate(png_directories):
        png_files = sorted(glob.glob(os.path.join(png_dir, "*.png")))
        print(f"Found {len(png_files)} PNG files in {png_dir}")

        if not png_files:
            continue

        all_png_files.append(png_files)
        idx = len(all_png_files) - 1

        with open(png_files[0], "rb") as f:
            img_data = f.read()

        image_widget = widgets.Image(
            value=img_data,
            format="png",
            width=image_width,
            height=image_height,
            layout=widgets.Layout(
                width=f"{image_width}px",
                height=f"{image_height}px",
                margin="0px"
            )
        )
        all_widgets.append(image_widget)

        label = widgets.HTML(
            value=f"<div style='font-size:11px; color:#444;'>0/{len(png_files)-1}</div>",
            layout=widgets.Layout(margin="0px")
        )
        all_labels.append(label)

        slider = widgets.IntSlider(
            value=0,
            min=0,
            max=len(png_files)-1,
            step=1,
            description="",
            readout=True,
            continuous_update=False,
            layout=widgets.Layout(width=f"{image_width}px", margin="0px")
        )
        all_sliders.append(slider)

        title = widgets.HTML(
            value=f"<div style='font-size:12px; font-weight:600; text-align:center;'>{simulation_names[i]}</div>",
            layout=widgets.Layout(margin="0px 0px 2px 0px")
        )

        def make_update_function(local_idx):
            def update_image(change):
                frame_idx = change["new"]
                files = all_png_files[local_idx]
                with open(files[frame_idx], "rb") as f:
                    all_widgets[local_idx].value = f.read()
                all_labels[local_idx].value = (
                    f"<div style='font-size:11px; color:#444;'>{frame_idx}/{len(files)-1}</div>"
                )
            return update_image

        slider.observe(make_update_function(idx), names="value")

        children = [title, all_labels[idx], image_widget]
        if show_individual_sliders:
            children.append(slider)

        sim_box = widgets.VBox(
            children,
            layout=widgets.Layout(
                align_items="center",
                justify_content="center",
                border="1px solid #D8D8D8",
                padding="4px",
                margin="0px",
                width=f"{image_width + 16}px"
            )
        )
        simulation_boxes.append(sim_box)

    if not all_png_files:
        return widgets.HTML("<b>No PNG files found.</b>")

    max_common_frame = min(len(files) for files in all_png_files) - 1

    master_slider = widgets.IntSlider(
        value=0,
        min=0,
        max=max_common_frame,
        step=1,
        description="Frame:",
        continuous_update=False,
        style={"description_width": "60px"},
        layout=widgets.Layout(width="700px")
    )

    play = widgets.Play(
        value=0,
        min=0,
        max=max_common_frame,
        step=1,
        interval=300
    )
    widgets.jslink((play, "value"), (master_slider, "value"))

    def sync_all(change):
        frame_idx = change["new"]
        for slider in all_sliders:
            if frame_idx <= slider.max:
                slider.value = frame_idx

    master_slider.observe(sync_all, names="value")

    controls = widgets.HBox(
        [widgets.HTML("<b>Timesteps:</b>"), play, master_slider],
        layout=widgets.Layout(
            align_items="center",
            justify_content="center",
            gap="10px",
            margin="0 0 10px 0"
        )
    )

    grid = widgets.GridBox(
        simulation_boxes,
        layout=widgets.Layout(
            grid_template_columns=f"repeat({ncols}, max-content)",
            grid_gap="8px 8px",
            justify_content="center"
        )
    )

    viewer = widgets.VBox(
        [controls, grid],
        layout=widgets.Layout(width="100%", align_items="center")
    )

    return viewer

In [7]:

PATH = "/scratch/flore0a/Dataset_test2/Cases"  # Folder where all sims are located
NAMES = Export_folders_from_results(PATH)
NAMES = NAMES[:-1]
name = name = [os.path.basename(path) for path in NAMES]

#how to remove the previos before /
png_directory = [d + "/png/u_t" for d in Directorio_results] # add the /png folder

quad_viewer = create_grid_viewer(png_directory,
    simulation_names=name,
    ncols=10,
    image_width=200,
    image_height=200,
    show_individual_sliders=True)
display(quad_viewer)

/scratch/flore0a/Dataset_test2/Cases/D2.5_V2
/scratch/flore0a/Dataset_test2/Cases/D2.5_V2p5
/scratch/flore0a/Dataset_test2/Cases/D2.5_V3
/scratch/flore0a/Dataset_test2/Cases/D2.5_V3p5
/scratch/flore0a/Dataset_test2/Cases/D2.5_V4
/scratch/flore0a/Dataset_test2/Cases/D2.5_V4p5
/scratch/flore0a/Dataset_test2/Cases/D2.5_V5
/scratch/flore0a/Dataset_test2/Cases/D2.5_V5p5
/scratch/flore0a/Dataset_test2/Cases/D2.5_V6
/scratch/flore0a/Dataset_test2/Cases/D2.5_V6p5
/scratch/flore0a/Dataset_test2/Cases/D2_V2
/scratch/flore0a/Dataset_test2/Cases/D2_V2p5
/scratch/flore0a/Dataset_test2/Cases/D2_V3
/scratch/flore0a/Dataset_test2/Cases/D2_V3p5
/scratch/flore0a/Dataset_test2/Cases/D2_V4
/scratch/flore0a/Dataset_test2/Cases/D2_V4p5
/scratch/flore0a/Dataset_test2/Cases/D2_V5
/scratch/flore0a/Dataset_test2/Cases/D2_V5p5
/scratch/flore0a/Dataset_test2/Cases/D2_V6
/scratch/flore0a/Dataset_test2/Cases/D2_V6p5
/scratch/flore0a/Dataset_test2/Cases/D3.5_V2
/scratch/flore0a/Dataset_test2/Cases/D3.5_V2p5
/scratch

In [8]:
import os
import glob
import math
from PIL import Image, ImageDraw

import os
import glob
import math
from PIL import Image, ImageDraw

def create_mosaic_gif(
    png_directories,
    output_file="mosaic.gif",
    simulation_names=None,
    ncols=10,
    tile_size=(120, 90),
    duration=250,
    keep_last_frame=True
):
    if simulation_names is None:
        simulation_names = [f"Sim {i+1}" for i in range(len(png_directories))]

    all_png_files = []
    valid_names = []

    for i, png_dir in enumerate(png_directories):
        png_files = sorted(glob.glob(os.path.join(png_dir, "*.png")))
        if png_files:
            all_png_files.append(png_files)
            valid_names.append(simulation_names[i])

    if not all_png_files:
        raise ValueError("No PNG files found.")

    n_sims = len(all_png_files)
    nrows = math.ceil(n_sims / ncols)
    max_frames = max(len(files) for files in all_png_files)

    tile_w, tile_h = tile_size
    title_h = 25
    full_tile_h = tile_h + title_h

    canvas_w = ncols * tile_w
    canvas_h = nrows * full_tile_h

    frames = []

    for frame_idx in range(max_frames):
        canvas = Image.new("RGB", (canvas_w, canvas_h), "white")
        draw = ImageDraw.Draw(canvas)

        for sim_idx, files in enumerate(all_png_files):
            row = sim_idx // ncols
            col = sim_idx % ncols
            x = col * tile_w
            y = row * full_tile_h

            if frame_idx < len(files):
                frame_to_use = frame_idx
                finished = False
            else:
                if keep_last_frame:
                    frame_to_use = len(files) - 1
                    finished = True
                else:
                    frame_to_use = None
                    finished = True

            # Title
            draw.text((x + 3, y + 2), valid_names[sim_idx][:15], fill="black")

            if frame_to_use is not None:
                img = Image.open(files[frame_to_use]).convert("RGB")
                img = img.resize((tile_w, tile_h))
                canvas.paste(img, (x, y + title_h))

                label = f"t={frame_to_use}"
                if finished:
                    label += " (last)"
                draw.text((x + 3, y + title_h - 10), label, fill="gray")
            else:
                blank = Image.new("RGB", (tile_w, tile_h), "lightgray")
                canvas.paste(blank, (x, y + title_h))
                draw.text((x + 10, y + title_h + tile_h // 2 - 5), "finished", fill="black")

        frames.append(canvas)

    frames[0].save(
        output_file,
        save_all=True,
        append_images=frames[1:],
        duration=duration,
        loop=0
    )

    print(f"Saved GIF to {output_file}")

In [9]:
#name
PATH = "/scratch/flore0a/Dataset_test2/Cases"  # Folder where all sims are located
NAMES = Export_folders_from_results(PATH)
NAMES = NAMES[:-1]
name = name = [os.path.basename(path) for path in NAMES]

#directory
png_directory = [d + "/png/u_t" for d in Directorio_results] # add the /png folder


create_mosaic_gif(
    png_directory,
    output_file="simulations_mosaic.gif",
    simulation_names=name,
    ncols=10,
    tile_size=(500, 400),
    duration=200
)

/scratch/flore0a/Dataset_test2/Cases/D2.5_V2
/scratch/flore0a/Dataset_test2/Cases/D2.5_V2p5
/scratch/flore0a/Dataset_test2/Cases/D2.5_V3
/scratch/flore0a/Dataset_test2/Cases/D2.5_V3p5
/scratch/flore0a/Dataset_test2/Cases/D2.5_V4
/scratch/flore0a/Dataset_test2/Cases/D2.5_V4p5
/scratch/flore0a/Dataset_test2/Cases/D2.5_V5
/scratch/flore0a/Dataset_test2/Cases/D2.5_V5p5
/scratch/flore0a/Dataset_test2/Cases/D2.5_V6
/scratch/flore0a/Dataset_test2/Cases/D2.5_V6p5
/scratch/flore0a/Dataset_test2/Cases/D2_V2
/scratch/flore0a/Dataset_test2/Cases/D2_V2p5
/scratch/flore0a/Dataset_test2/Cases/D2_V3
/scratch/flore0a/Dataset_test2/Cases/D2_V3p5
/scratch/flore0a/Dataset_test2/Cases/D2_V4
/scratch/flore0a/Dataset_test2/Cases/D2_V4p5
/scratch/flore0a/Dataset_test2/Cases/D2_V5
/scratch/flore0a/Dataset_test2/Cases/D2_V5p5
/scratch/flore0a/Dataset_test2/Cases/D2_V6
/scratch/flore0a/Dataset_test2/Cases/D2_V6p5
/scratch/flore0a/Dataset_test2/Cases/D3.5_V2
/scratch/flore0a/Dataset_test2/Cases/D3.5_V2p5
/scratch

In [11]:
import os
import glob
import math
import re
from PIL import Image, ImageDraw

def parse_case_name(case_path):
    """
    Extract D and V from folder name like:
    D12_V0p5 -> D=12, V=0.5
    """
    name = os.path.basename(case_path.rstrip("/"))
    m = re.match(r"D(\d+)_V(\d+(?:p\d+)?)$", name)
    if not m:
        return None

    d_val = int(m.group(1))
    v_str = m.group(2).replace("p", ".")
    v_val = float(v_str)
    return d_val, v_val

def build_case_matrix(base_dir):
    """
    Finds all folders matching D*_V* and arranges them into a matrix:
    rows = sorted D values
    cols = sorted V values
    """
    case_dirs = sorted(glob.glob(os.path.join(base_dir, "D*_V*")))

    parsed = []
    d_values = set()
    v_values = set()

    for cdir in case_dirs:
        parsed_case = parse_case_name(cdir)
        if parsed_case is None:
            continue
        d_val, v_val = parsed_case
        parsed.append((d_val, v_val, cdir))
        d_values.add(d_val)
        v_values.add(v_val)

    d_values = sorted(d_values)
    v_values = sorted(v_values)

    # dictionary for quick lookup
    case_dict = {(d, v): path for d, v, path in parsed}

    return d_values, v_values, case_dict


def create_mosaic_gif_matrix(
    base_dir,
    output_file="mosaic_matrix.gif",
    tile_size=(140, 100),
    duration=250,
    keep_last_frame=True,
    png_pattern="*.png",
    png_subdir=os.path.join("png", "u_t")
):
    d_values, v_values, case_dict = build_case_matrix(base_dir)

    if not d_values or not v_values:
        raise ValueError("No valid D*_V* case folders found.")

    nrows = len(d_values)
    ncols = len(v_values)

    # Load PNG lists per case
    all_cases = {}
    max_frames = 0

    for d in d_values:
            for v in v_values:
                case_path = case_dict.get((d, v), None)
                if case_path is None:
                    all_cases[(d, v)] = None
                    continue
    
                png_dir = os.path.join(case_path, png_subdir)
    
                if not os.path.isdir(png_dir):
                    all_cases[(d, v)] = []
                    continue
    
                png_files = sorted(glob.glob(os.path.join(png_dir, png_pattern)))
                if png_files:
                    all_cases[(d, v)] = png_files
                    max_frames = max(max_frames, len(png_files))
                else:
                    all_cases[(d, v)] = []
    
    if max_frames == 0:
            raise ValueError(f"No PNG files found under subdirectory '{png_subdir}'.")


    tile_w, tile_h = tile_size

    # Extra space:
    left_label_w = 70   # space for row labels (D values)
    top_label_h = 30    # space for column labels (V values)
    title_h = 18        # per tile title
    time_h = 15         # per tile time label

    full_tile_h = title_h + tile_h + time_h
    canvas_w = left_label_w + ncols * tile_w
    canvas_h = top_label_h + nrows * full_tile_h

    frames = []

    for frame_idx in range(max_frames):
        canvas = Image.new("RGB", (canvas_w, canvas_h), "white")
        draw = ImageDraw.Draw(canvas)

        # Column headers: V values
        for col, v in enumerate(v_values):
            x = left_label_w + col * tile_w
            draw.text((x + 10, 5), f"V={v:g}", fill="black")

        # Row headers: D values
        for row, d in enumerate(d_values):
            y = top_label_h + row * full_tile_h
            draw.text((10, y + title_h + tile_h // 2 - 5), f"D={d}", fill="black")

        # Fill tiles
        for row, d in enumerate(d_values):
            for col, v in enumerate(v_values):
                x = left_label_w + col * tile_w
                y = top_label_h + row * full_tile_h

                case_name = f"D{d}_V{str(v).replace('.', 'p')}"
                files = all_cases[(d, v)]

                # Small case title
                draw.text((x + 3, y + 1), case_name, fill="black")

                if files is None:
                    # Folder missing entirely
                    blank = Image.new("RGB", (tile_w, tile_h), "lightgray")
                    canvas.paste(blank, (x, y + title_h))
                    draw.text((x + 10, y + title_h + tile_h // 2 - 5), "missing", fill="black")
                    continue

                if len(files) == 0:
                    # Folder exists but no png
                    blank = Image.new("RGB", (tile_w, tile_h), "gainsboro")
                    canvas.paste(blank, (x, y + title_h))
                    draw.text((x + 10, y + title_h + tile_h // 2 - 5), "no png", fill="black")
                    continue

                if frame_idx < len(files):
                    frame_to_use = frame_idx
                    finished = False
                else:
                    if keep_last_frame:
                        frame_to_use = len(files) - 1
                        finished = True
                    else:
                        frame_to_use = None
                        finished = True

                if frame_to_use is not None:
                    img = Image.open(files[frame_to_use]).convert("RGB")
                    img = img.resize((tile_w, tile_h))
                    canvas.paste(img, (x, y + title_h))

                    label = f"t={frame_to_use}"
                    if finished:
                        label += " (last)"
                    draw.text((x + 3, y + title_h + tile_h + 1), label, fill="gray")
                else:
                    blank = Image.new("RGB", (tile_w, tile_h), "lightgray")
                    canvas.paste(blank, (x, y + title_h))
                    draw.text((x + 10, y + title_h + tile_h // 2 - 5), "finished", fill="black")

        frames.append(canvas)

    frames[0].save(
        output_file,
        save_all=True,
        append_images=frames[1:],
        duration=duration,
        loop=0
    )

    print(f"Saved GIF to {output_file}")
    print(f"Matrix size: {nrows} rows (D) x {ncols} cols (V)")
    print(f"D values: {d_values}")
    print(f"V values: {v_values}")




In [12]:
create_mosaic_gif_matrix(
    base_dir="/scratch/flore0a/Dataset_test2/Cases",
    output_file="/scratch/flore0a/Dataset_test2/Cases/mosaic_matrix.gif",
    tile_size=(120, 90),
    duration=250,
    keep_last_frame=True,
    png_pattern="*.png",
    png_subdir=os.path.join("Simulation2D", "merged", "png", "u_t")
)

Saved GIF to /scratch/flore0a/Dataset_test2/Cases/mosaic_matrix.gif
Matrix size: 8 rows (D) x 10 cols (V)
D values: [2, 3, 4, 5, 6, 7, 8, 9]
V values: [2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0, 6.5]


PARAVIEW


In [17]:


def Create_view_white(image_size=(2048, 2048)):
    try:
        renderView = pv.GetActiveViewOrCreate('RenderView')
        renderView.ViewSize = list(image_size)
        renderView.Background = [0,0,0]  # Fondo negro
        renderView.BackgroundColorMode = 'Single Color'  
        renderView.OrientationAxesVisibility = 0  # Sin ejes
        renderView.CameraParallelProjection = 1
        ##pv.LoadPalette(paletteName='WhiteBackground')
        return renderView

    except Exception as e:
        print(f" Error en BackgroundWhite: {e}")
    
def calculate_perfect_parallel_scale(width, height, image_size, margin_factor=0):
    """Calcula escala paralela perfecta para llenar la imagen"""
    geom_ratio = width / height
    image_ratio = image_size[0] / image_size[1]
    
    if geom_ratio > image_ratio:
        # Geometría más ancha que imagen → width limita
        scale = (width / 2) * (1 + margin_factor)
        limiting_dimension = "WIDTH"
    else:
        # Geometría más alta que imagen → height limita  
        scale = (height / 2) * (1 + margin_factor)
        limiting_dimension = "HEIGHT"
    
    return scale  

def Camara_position(reader, renderView,image_size=(2048, 2048)):
    try:

        t0 = reader.TimestepValues[0]
        pv.UpdatePipeline(time=t0, proxy=reader) # fix the camara to the initial one before any resample data
                
        # Obtener bounds sin mostrar
        data_info = reader.GetDataInformation()
        bounds = data_info.GetBounds()

    
        width = bounds[1] - bounds[0]
        height = bounds[3] - bounds[2]
        center_x = (bounds[0] + bounds[1]) / 2
        center_y = (bounds[2] + bounds[3]) / 2
        center_z = (bounds[4] + bounds[5]) / 2
        
        # Configurar cámara perfecta
        camera = renderView.GetActiveCamera()
        camera.SetPosition([center_x, center_y, center_z + max(width, height) * 1.5])
        camera.SetFocalPoint([center_x, center_y, center_z])
        camera.SetViewUp([0, 1, 0])
        
        

        # ¡CLAVE! Escala paralela perfecta
        optimal_scale = calculate_perfect_parallel_scale(width, height, image_size, margin_factor=0.0)
        camera.SetParallelScale(optimal_scale)
        
        return camera

    except Exception as e:
        print(f"❌ Error en Camara_position: {e}")
        return None
    
############
import paraview.simple as pv
import glob
import os


def export_one(path,variable_input='concentration'):
    print(f"🚀 Exporting PNGs from: {path}")
    export_png_from_vtu(path, variable=variable_input, min_range=0.3, max_range=0.9)
    print(f"✅ Done: {path}")
    
        
def export_png_from_vtu(vtu_file, variable = 'concentration',min_range=0, max_range=1.0):
    pv.ResetSession()
    renderView = Create_view_white()

   #look for the pvtu files
    if os.path.isdir(vtu_file):
        files = sorted(glob.glob(os.path.join(vtu_file, "Output_t*.vtu"))) ## change to vtu
    else:
        files = sorted(glob.glob(vtu_file))
        
    # Leer archivo VTU
    reader = pv.XMLUnstructuredGridReader(FileName=files) 
    reader.PointArrayStatus = [variable,"lsf"]
    #reader.UpdatePipeline()
    reader.TimeArray = "None"

    #display = pv.Show(reader, renderView)
    #pv.ColorBy(display, ('POINTS', variable))
    #display.Representation = 'Surface'

    tmp_display = pv.Show(reader, renderView)
    pv.Render()
    
    camara = Camara_position(reader,renderView) ## center the image
    pv.Hide(tmp_display, renderView)
    pv.Render()

    #save pngs:
    out_directory = os.path.join(vtu_file, "png")
    os.makedirs(out_directory, exist_ok=True)

    contour = pv.Contour(Input=reader)
    contour.ContourBy = ['POINTS', 'lsf']
    contour.Isosurfaces = [0.0]
    display_lsf = pv.Show(contour, renderView)
    #pv.ColorBy(display_lsf, None)
    display_lsf.ColorArrayName = ['POINTS', '']
    display_lsf.Representation = 'Surface'
    display_lsf.DiffuseColor = [0, 0, 0]
    display_lsf.LineWidth = 2
    pv.Hide(display_lsf, renderView) 

    thresh = pv.Threshold(Input=reader)
    thresh.Scalars = ['POINTS', 'lsf']
    thresh.LowerThreshold = 0
    thresh.ThresholdMethod = 'Below Lower Threshold'
    display_th = pv.Show(thresh, renderView)
    display_th.Representation = 'Surface'
    pv.Hide(display_th, renderView)

    resample = pv.ResampleWithDataset(SourceDataArrays=reader, DestinationMesh=thresh)
    resample.CellLocator = 'Static Cell Locator'
    display_resample = pv.Show(resample, renderView)
    display_resample.ColorArrayName = ['POINTS', variable]


    #Create after have your final display : it is display_resample
    lut = pv.GetColorTransferFunction(variable, display_resample)
    lut.ApplyPreset('Inferno (matplotlib)', True)
    lut.InvertTransferFunction()
    lut.RescaleTransferFunction(min_range, max_range)

    # Guardar PNGs
    for idx, t in enumerate(reader.TimestepValues):
        print (f" Saving timestep {idx} / {len(reader.TimestepValues)}")
        renderView.ViewTime = t
        pv.Render()
        pv.SaveScreenshot(os.path.join(out_directory, f"{idx:03d}.png"), 
                        renderView,
                        #quality=100,
                        ImageResolution=renderView.ViewSize, 
                        TransparentBackground=1)


In [18]:
export_one('/scratch/flore0a/Dataset_test1/Cases/D6_V5/Simulation2D/merged', variable_input="u_t")

🚀 Exporting PNGs from: /scratch/flore0a/Dataset_test1/Cases/D6_V5/Simulation2D/merged


(1378.131s) [paraview        ]vtkSMParaViewPipelineCo:506    ERR| vtkSMParaViewPipelineControllerWithRendering (0x5642011a7e00): Invalid producer (0x564206cb2e00) or outputPort (0)
(1378.204s) [paraview        ]vtkSMParaViewPipelineCo:506    ERR| vtkSMParaViewPipelineControllerWithRendering (0x5642011a7e00): Invalid producer (0x5641de1bdc50) or outputPort (0)


 Saving timestep 0 / 24
 Saving timestep 1 / 24
 Saving timestep 2 / 24
 Saving timestep 3 / 24
 Saving timestep 4 / 24
 Saving timestep 5 / 24
 Saving timestep 6 / 24
 Saving timestep 7 / 24
 Saving timestep 8 / 24
 Saving timestep 9 / 24
 Saving timestep 10 / 24
 Saving timestep 11 / 24
 Saving timestep 12 / 24
 Saving timestep 13 / 24
 Saving timestep 14 / 24
 Saving timestep 15 / 24
 Saving timestep 16 / 24
 Saving timestep 17 / 24
 Saving timestep 18 / 24
 Saving timestep 19 / 24
 Saving timestep 20 / 24
 Saving timestep 21 / 24
 Saving timestep 22 / 24
 Saving timestep 23 / 24
✅ Done: /scratch/flore0a/Dataset_test1/Cases/D6_V5/Simulation2D/merged
